# Data Preprocessing Tools

**The big picture:** the data has 10 customers. For each one we know their **Country**, **Age** and **Salary**, and whether they **Purchased** a product (Yes/No). Nothing is predicted here yet. The goal is to *clean and prepare* the data so a model can use it later.

Think of it like preparing ingredients before cooking: fill in what's missing, turn words into numbers, set some aside for tasting later, and cut everything to a similar size. The steps are:

> **fill gaps → turn words into numbers → split into training and test sets → put numbers on a similar scale**

### Before you start
- **Read first:** Nothing — this is the starting point.
- **New words in this notebook:** *missing values*, *imputation*, *one-hot encoding*, *label encoding*, *train/test split*, *feature scaling*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

## Importing the libraries

- **pandas** reads and handles tables, like Excel in Python.
- **numpy** does fast maths on lists of numbers. Here it also gives us `np.nan`, which means "missing value".
- **matplotlib** draws charts. It is imported out of habit but not used in this notebook.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

- Reads `Data.csv` into a table.
- **X** = every column except the last (Country, Age, Salary). These are the **inputs**, also called *features*.
- **y** = the last column, *Purchased*. This is what a model would later try to **predict**.
- `.values` turns the table into a plain grid of values that scikit-learn (the ML library) can work with.

When you read the `print(X)` output, notice two `nan` values: the German customer's salary (row 5) and the Spanish customer's age (row 7) are missing. `print(y)` shows the Yes/No answers, still as words.

In [0]:
dataset = pd.read_csv('Data.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [3]:
print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 nan]
 ['France' 35.0 58000.0]
 ['Spain' nan 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


In [4]:
print(y)

['No' 'Yes' 'No' 'No' 'Yes' 'Yes' 'No' 'Yes' 'No' 'Yes']


## Taking care of missing data

Most models crash or give nonsense when a value is missing. Deleting those rows would throw away useful data (and with only 10 rows, every row counts). Instead we **fill each gap with the average of its column**.

- `missing_values=np.nan` → "look for empty cells".
- `strategy='mean'` → "fill them with the column's average".
- `X[:, 1:3]` → only the Age and Salary columns (positions 1 and 2, counting from 0). Country is skipped because you can't average words.
- `fit` works out the averages. `transform` writes them into the gaps.

In the second `print(X)` the gaps are filled: the missing salary became **63,777.78** (the average of the 9 known salaries) and the missing age became **38.78** (the average of the 9 known ages).

In [0]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
imputer.fit(X[:, 1:3])
X[:, 1:3] = imputer.transform(X[:, 1:3])

In [6]:
print(X)

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['Spain' 38.0 61000.0]
 ['Germany' 40.0 63777.77777777778]
 ['France' 35.0 58000.0]
 ['Spain' 38.77777777777778 52000.0]
 ['France' 48.0 79000.0]
 ['Germany' 50.0 83000.0]
 ['France' 37.0 67000.0]]


## Encoding categorical data

"Categorical" data is data made of labels or categories (like country names or Yes/No) rather than measurements. Models only do maths on numbers, so these labels must be turned into numbers. There are two cases below: the inputs (Country) and the answer (Purchased).

### Encoding the Independent Variable

The *independent variables* are the inputs (X). Here the one that needs encoding is **Country**.

Replacing countries with 0, 1 and 2 would wrongly suggest an order, as if Spain were "bigger" than France. **One-hot encoding** instead replaces the Country column with **three yes/no columns**, one per country:

| Country | France | Germany | Spain |
|---|---|---|---|
| France | 1 | 0 | 0 |
| Germany | 0 | 1 | 0 |
| Spain | 0 | 0 | 1 |

- `[0]` means "apply this to column 0", which is Country.
- `remainder='passthrough'` means "leave the other columns (Age, Salary) as they are".
- `np.array(...)` makes sure the result is a normal numpy grid.

In the printout, the first three columns are the new country columns and the last two are Age and Salary. For example, the first row `1.0 0.0 0.0 44.0 72000.0` is a 44-year-old from France earning 72,000.

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

In [8]:
print(X)

[[1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [0.0 1.0 0.0 30.0 54000.0]
 [0.0 0.0 1.0 38.0 61000.0]
 [0.0 1.0 0.0 40.0 63777.77777777778]
 [1.0 0.0 0.0 35.0 58000.0]
 [0.0 0.0 1.0 38.77777777777778 52000.0]
 [1.0 0.0 0.0 48.0 79000.0]
 [0.0 1.0 0.0 50.0 83000.0]
 [1.0 0.0 0.0 37.0 67000.0]]


### Encoding the Dependent Variable

The *dependent variable* is the answer we want to predict (y), here *Purchased*.

- **LabelEncoder** simply turns each label into a number: **No → 0** and **Yes → 1** (it goes alphabetically).
- One-hot encoding isn't needed here. With only two options, a single 0/1 column says everything.

The printout `[0 1 0 0 1 1 0 1 0 1]` is the same Yes/No list as before, just in numbers.

In [0]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [10]:
print(y)

[0 1 0 0 1 1 0 1 0 1]


## Splitting the dataset into the Training set and Test set

- `test_size = 0.2` → **80% of the rows (8 customers)** go into the *training set*, which the model will learn from.
- **20% of the rows (2 customers)** go into the *test set*, held back to check the model afterwards.
- `random_state = 1` makes the shuffle come out the same every time you run it.
- The four printouts show the result: 8 rows in `X_train` with their 8 answers in `y_train`, and 2 rows in `X_test` with their 2 answers in `y_test`.

**Why?** Testing on data the model has never seen shows whether it actually learned the pattern rather than memorising the answers. It's like giving a student an exam with new questions, not the ones they practised on.

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

In [12]:
print(X_train)

[[0.0 0.0 1.0 38.77777777777778 52000.0]
 [0.0 1.0 0.0 40.0 63777.77777777778]
 [1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 38.0 61000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [1.0 0.0 0.0 48.0 79000.0]
 [0.0 1.0 0.0 50.0 83000.0]
 [1.0 0.0 0.0 35.0 58000.0]]


In [13]:
print(X_test)

[[0.0 1.0 0.0 30.0 54000.0]
 [1.0 0.0 0.0 37.0 67000.0]]


In [14]:
print(y_train)

[0 1 0 0 1 1 0 1]


In [15]:
print(y_test)

[0 1]


## Feature Scaling

Age is around 30-50, but Salary is around 50,000-80,000. Some models would think Salary matters far more just because its numbers are bigger. **Feature scaling** puts both columns on the same scale.

- **StandardScaler** (standardisation) shifts and stretches each column so its average becomes 0 and most values fall roughly between -2 and +2. In short: *(value − average) ÷ spread*.
- `X_train[:, 3:]` → only the Age and Salary columns (position 3 onwards). The three country columns are **not** scaled: they are already 0 or 1, and scaling them would hide which country each row is.
- `fit_transform` on the training set works out the average and spread **from the training data only**, then applies them.
- `transform` on the test set reuses those **same** numbers. It does *not* fit again.

**Why fit only on training data?** The test set is meant to be "unseen". If its values helped set the scale, a bit of information about the test would leak into training.

In the printouts, Age and Salary are now small numbers around 0. A negative value means "below the training average", a positive one means "above it".

**In one sentence:** load customer data, fill the missing ages and salaries with averages, turn countries into yes/no columns and Yes/No into 1/0, keep 20% aside for testing, then put Age and Salary on the same scale so no column dominates just because its numbers are bigger.

In [0]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train[:, 3:] = sc.fit_transform(X_train[:, 3:])
X_test[:, 3:] = sc.transform(X_test[:, 3:])

In [17]:
print(X_train)

[[0.0 0.0 1.0 -0.19159184384578545 -1.0781259408412425]
 [0.0 1.0 0.0 -0.014117293757057777 -0.07013167641635372]
 [1.0 0.0 0.0 0.566708506533324 0.633562432710455]
 [0.0 0.0 1.0 -0.30453019390224867 -0.30786617274297867]
 [0.0 0.0 1.0 -1.9018011447007988 -1.420463615551582]
 [1.0 0.0 0.0 1.1475343068237058 1.232653363453549]
 [0.0 1.0 0.0 1.4379472069688968 1.5749910381638885]
 [1.0 0.0 0.0 -0.7401495441200351 -0.5646194287757332]]


In [18]:
print(X_test)

[[0.0 1.0 0.0 -1.4661817944830124 -0.9069571034860727]
 [1.0 0.0 0.0 -0.44973664397484414 0.2056403393225306]]


## Common mistakes to avoid
- **Scaling before splitting, or calling `fit_transform` on the test set.** That lets the test rows influence the average and spread, so the test is no longer truly "unseen" (this is called *data leakage*). Split first, `fit_transform` on `X_train`, and only `transform` on `X_test`.
- **Scaling the one-hot country columns too.** If you scale all of `X_train` instead of `X_train[:, 3:]`, the clean 0/1 country flags turn into odd decimals and lose their simple meaning. Only scale the real number columns (Age and Salary).
- **Re-running a cell that changes `X` in place.** Running the one-hot encoding cell a second time encodes column 0 again, but column 0 is now the France flag, not Country, so X quietly becomes nonsense. Running the scaling cell twice scales already-scaled numbers. If in doubt, use *Restart & Run All*.
- **Getting the column slice wrong.** In Python, `X[:, 1:3]` means columns 1 and 2 — the end number is *not* included. Writing `X[:, 1:2]` would fill only Age and leave Salary's gap as `nan`.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. In the missing-data cell, change `strategy='mean'` to `strategy='median'` and re-run from the top. What to notice: the gaps are now filled with the middle value instead of the average — the missing age becomes **38.0** and the missing salary **61000.0**. The median is less pulled around by unusually high or low values.
2. After the `LabelEncoder` cell, add a new cell with `print(le.classes_)`. What to notice: it prints `['No' 'Yes']`, the labels in alphabetical order — position 0 is No and position 1 is Yes, which is exactly why No became 0 and Yes became 1.
3. In the split cell, change `random_state = 1` to `random_state = 0` and re-run from the top. What to notice: different customers end up in `X_test`, and the scaled Age and Salary numbers change too, because the average and spread are now worked out from a different set of 8 training customers.
4. After the feature-scaling cell, add `print(sc.mean_)` and `print(sc.scale_)`. What to notice: these are the training-set average and spread for Age and Salary that were used to scale *both* the training and the test rows. Salary's spread is in the thousands while Age's is only a few years — that size gap is exactly what scaling removes.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. Why not just replace France, Germany and Spain with 0, 1 and 2?**

<details><summary>Show answer</summary>

Numbers imply an order and size, so the model would think Spain is "bigger" than France, or that Germany sits halfway between them. One-hot encoding gives each country its own yes/no column, so no fake order is created.

</details>

**2. In the second `print(X)`, the missing salary became 63777.78. Where does that number come from?**

<details><summary>Show answer</summary>

It is the average of the 9 salaries that were known. `SimpleImputer` with `strategy='mean'` worked out each column's average in `fit` and wrote it into the gap in `transform`.

</details>

**3. Why does the test set use `sc.transform` but the training set uses `sc.fit_transform`?**

<details><summary>Show answer</summary>

`fit` learns the average and spread, and that must come from the training data only. The test set is then scaled with those same numbers, so it is treated exactly like brand-new data the model would meet later.

</details>

**4. What would happen if you changed the imputer to work on `X[:, 0:3]` instead of `X[:, 1:3]`?**

<details><summary>Show answer</summary>

It would fail with an error, because column 0 holds country names and you can't take the average of words. That is why the code starts at column 1.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)